# TP NOTÉ S3 — REGRESSION ENGINEERING CHALLENGE
## 4e année Option Développeur — 3 h — Ames Housing, Iowa (USA)

**Rendu :** un notebook Jupyter entièrement exécuté. Toute décision doit être justifiée par vos propres résultats.

**Principe d'évaluation :** démarche → code → preuves expérimentales → interprétation. Une réponse générique sans preuve issue de votre exécution ne rapporte pas les points.

## Contexte & dataset — 5 pts

Vous intervenez pour une équipe qui doit estimer le **prix de vente de logements à Ames, Iowa**. Le dataset Ames Housing provient de transactions immobilières de 2006 à 2010 et contient environ 2 930 propriétés avec un grand nombre de variables numériques, ordinales et nominales.

**Cible : `SalePrice`.**

Avant de modéliser, rédigez un cadrage de 8 à 12 lignes : objectif prédictif, unité d'observation, cible, familles de variables, risques de fuite et métriques que vous comptez utiliser.

Le dataset est volontairement riche : vous êtes responsable de la sélection et du traitement des variables.

In [1]:
# Votre travail expérimental ici
import pandas as pd

df = pd.read_csv("https://jse.amstat.org/v19n3/decock/AmesHousing.txt", sep="\t")
print(df.shape)
df.head()

(2930, 82)


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


## Audit orienté modélisation — 15 pts

Sans produire une EDA encyclopédique, établissez les diagnostics nécessaires à une régression fiable.

Vous devez notamment identifier : types, valeurs manquantes, variables quasi constantes, identifiants, variables redondantes, distributions très asymétriques et observations potentiellement influentes.

**Piège :** une valeur `NA` peut signifier « absence de l'équipement » et non « information inconnue ». Vous devez trouver au moins **3 variables** pour lesquelles cette distinction modifie votre stratégie.

Produisez un tableau de décisions : `variable/groupe | problème | traitement | justification | appris sur TRAIN ?`.

In [2]:
# Votre travail expérimental ici
# Types
print(df.dtypes.value_counts())

# Valeurs manquantes
na = df.isna().sum()
pct = (na / len(df) * 100).round(1)
audit_na = pd.DataFrame({'nb_na': na, 'pct_na': pct})
audit_na = audit_na[audit_na['nb_na'] > 0].sort_values('pct_na', ascending=False)
print(audit_na)

# Quasi constantes
for col in df.select_dtypes(include='object').columns:
    top = df[col].value_counts(normalize=True, dropna=False).iloc[0] * 100
    if top > 95:
        print(col, round(top,1), '%')

# Identifiants à exclure
print(df[['Order','PID']].nunique())

str        43
int64      28
float64    11
Name: count, dtype: int64
                nb_na  pct_na
Pool QC          2917    99.6
Misc Feature     2824    96.4
Alley            2732    93.2
Fence            2358    80.5
Mas Vnr Type     1775    60.6
Fireplace Qu     1422    48.5
Lot Frontage      490    16.7
Garage Qual       159     5.4
Garage Yr Blt     159     5.4
Garage Type       157     5.4
Garage Finish     159     5.4
Garage Cond       159     5.4
Bsmt Exposure      83     2.8
BsmtFin Type 2     81     2.8
Bsmt Cond          80     2.7
Bsmt Qual          80     2.7
BsmtFin Type 1     80     2.7
Mas Vnr Area       23     0.8
Bsmt Full Bath      2     0.1
Bsmt Half Bath      2     0.1
BsmtFin SF 1        1     0.0
BsmtFin SF 2        1     0.0
Electrical          1     0.0
Total Bsmt SF       1     0.0
Bsmt Unf SF         1     0.0
Garage Area         1     0.0
Garage Cars         1     0.0
Street 99.6 %
Utilities 99.9 %
Land Slope 95.2 %
Condition 2 99.0 %
Roof Matl 98.5 %
Heating

C:\Users\jordan.jimenez\AppData\Local\Temp\ipykernel_13136\1697066582.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


## Protocole expérimental — 10 pts

Construisez TRAIN / VALIDATION / TEST. Le TEST doit rester invisible jusqu'à la décision finale.

Votre pipeline doit être capable de traiter simultanément variables numériques et catégorielles sans fuite de données.

**Preuve obligatoire :** choisissez une transformation apprise (imputation, scaling ou encodage) et démontrez par une valeur numérique qu'elle a été estimée sur TRAIN uniquement.

In [ ]:
# Votre travail expérimental ici


## Baseline et modèle linéaire — 10 pts

Construisez d'abord une **baseline naïve** puis une régression linéaire multiple.

Comparez-les sur VALIDATION avec **MSE, RMSE et R²**. Analysez au moins un graphique de résidus.

Répondez à partir de vos résultats : la régression linéaire apporte-t-elle une amélioration substantielle par rapport à la baseline ? Quelles structures restent visibles dans les résidus ?

In [ ]:
# Votre travail expérimental ici


## Polynomial challenge — 15 pts

Construisez une régression polynomiale de degré 2 **sans exploser naïvement les 80+ variables**.

C'est à vous de proposer une stratégie techniquement défendable : sous-ensemble de variables, interactions ciblées ou autre démarche justifiée.

Comparez TRAIN et VALIDATION. Quantifiez le nombre de features avant/après transformation.

**Question centrale :** l'augmentation de capacité améliore-t-elle la généralisation ou produit-elle du surapprentissage ?

In [ ]:
# Votre travail expérimental ici


## Ridge vs Lasso — 20 pts

Construisez Ridge et Lasso sur un pipeline cohérent. Étudiez plusieurs ordres de grandeur de `alpha`.

Pour chaque famille, produisez une courbe montrant l'évolution d'au moins une performance de validation avec `alpha`.

Pour Lasso, mesurez le nombre de coefficients exactement nuls. Pour Ridge, étudiez la norme des coefficients.

Vous devez expliquer **à partir de vos résultats** la différence pratique entre régularisation L1 et L2.

In [ ]:
# Votre travail expérimental ici


## Stress test — 10 pts

Votre modèle final doit subir un stress test.

Identifiez les **10 plus grandes erreurs absolues** sur VALIDATION. Pour au moins 3 maisons, comparez leurs caractéristiques à celles d'observations typiques et proposez une hypothèse expliquant l'erreur.

Une simple liste de prix réels/prédits ne suffit pas.

In [ ]:
# Votre travail expérimental ici


## Décision finale & TEST — 10 pts

Figez votre modèle et vos hyperparamètres **avant** d'ouvrir TEST. Évaluez une seule fois sur TEST.

Présentez MSE, RMSE, R² et une comparaison VALIDATION/TEST. Expliquez si la généralisation est cohérente avec ce que vous aviez anticipé.

Terminez par une recommandation technique de 10 lignes maximum.

In [ ]:
# Votre travail expérimental ici


## Reproductibilité & intégrité — 5 pts

### Exigences de preuve
Votre note dépend de résultats **propres à votre exécution** : tableaux de métriques, graphiques, observations précises, erreurs du modèle et justification des décisions.  
Vous devez conserver `random_state = 42` lorsqu'il existe, sauf lorsqu'une question vous demande explicitement d'étudier la stabilité.

### Ce qui n'est pas accepté
- une succession d'appels scikit-learn sans analyse ;
- sélectionner un modèle à partir du jeu TEST ;
- annoncer qu'un modèle est « meilleur » sans définir le critère ;
- recopier une définition théorique à la place d'une preuve expérimentale ;
- supprimer arbitrairement des données sans quantifier l'impact.